In [ ]:
# Cell 1
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED_DIGIT = 5      
ID_SUM     = 26      
LAST_NAME  = "Lancion"   
STUDENT_ID = "TUPM-26-1845"   
DATA_PATH   = "data/secom.data"
LABELS_PATH = "data/secom_labels.data"
MAX_MISSING_PCT = 50.0   
OUT_DIR = "ex3_robustness_sensitivity"   
# ------------------------------------------------------
print("Student:", LAST_NAME, "| ID:", STUDENT_ID)
print("SEED_DIGIT =", SEED_DIGIT, "| ID_SUM =", ID_SUM)
X_df = pd.read_csv(DATA_PATH, sep=r"\s+", header=None).apply(pd.to_numeric, errors="coerce")
y = pd.read_csv(LABELS_PATH, sep=r"\s+", header=None, usecols=[0], names=["label"])["label"].to_numpy()
X_df.index.name = "record_id"          # original 0-based record identifier
X = X_df.to_numpy(dtype=float)
assert X.shape[0] == len(y), "data / label row mismatch"
fname = lambda i: f"F{i:03d}"

n_obs = (~np.isnan(X)).sum(axis=0)
missing_pct = np.isnan(X).mean(axis=0) * 100
constant = (n_obs == 0) | (np.nanmin(X, axis=0) == np.nanmax(X, axis=0))
usable_idx = np.where(~constant & (missing_pct <= MAX_MISSING_PCT))[0]
print("Matrix:", X.shape, "| Pass(-1):", (y == -1).sum(), "| Fail(1):", (y == 1).sum())
print("Features with missing:", (missing_pct > 0).sum(), "| constant/empty:", constant.sum(), "| usable:", len(usable_idx))
os.makedirs(OUT_DIR, exist_ok=True)

Student: Lancion | ID: TUPM-26-1845
SEED_DIGIT = 5 | ID_SUM = 26
Matrix: (1567, 590) | Pass(-1): 1463 | Fail(1): 104
Features with missing: 538 | constant/empty: 116 | usable: 446


In [ ]:
# Cell 2
fail_m, pass_m = (y == 1), (y == -1)
ok_cnt = ((~np.isnan(X[fail_m])).sum(0) >= 10) & ((~np.isnan(X[pass_m])).sum(0) >= 10) & ~constant
cand = np.where(ok_cnt)[0]
sel3 = np.sort(np.random.default_rng(SEED_DIGIT).choice(cand, size=5, replace=False))
cols3 = [fname(i) for i in sel3]
B = 200 + 10 * SEED_DIGIT
print("Selected:", cols3, "| Bootstrap repetitions:", B)


def boot_diff(fail, pas, B, rng):
    """Resample with replacement independently within each outcome group."""
    fi = rng.integers(0, len(fail), size=(B, len(fail)))
    pi = rng.integers(0, len(pas),  size=(B, len(pas)))
    return fail[fi].mean(axis=1) - pas[pi].mean(axis=1)

def run_pipeline(seed):
    rng = np.random.default_rng(seed)
    out = []
    for strategy in ["available_case", "median_imputation"]:
        for idx in sel3:
            col = X[:, idx]
            if strategy == "median_imputation":
                col = np.where(np.isnan(col), np.nanmedian(col), col)   # full-dataset median
            f, p = col[fail_m], col[pass_m]
            f, p = f[~np.isnan(f)], p[~np.isnan(p)]                      # drops NaN (available-case)
            d = f.mean() - p.mean()                                      # Step 3: Fail - Pass
            bd = boot_diff(f, p, B, rng)
            lo, hi = np.percentile(bd, [2.5, 97.5])
            out.append({"feature": fname(idx), "strategy": strategy, "n_fail": len(f), "n_pass": len(p),
                        "mean_diff": d, "boot_mean_diff": bd.mean(), "ci_low": lo, "ci_high": hi,
                        "ci_excludes_zero": not (lo <= 0 <= hi)})
    r = pd.DataFrame(out)
    r["abs_rank"] = r.groupby("strategy")["mean_diff"].transform(lambda s: s.abs().rank(ascending=False).astype(int))
    return r

res = run_pipeline(SEED_DIGIT)
display(res)


a = res[res.strategy == "available_case"].set_index("feature")
m = res[res.strategy == "median_imputation"].set_index("feature")
comp3 = pd.DataFrame({
    "diff_available": a.mean_diff, "diff_imputed": m.mean_diff,
    "change_in_diff": m.mean_diff - a.mean_diff,
    "sign_changed": np.sign(a.mean_diff) != np.sign(m.mean_diff),
    "ci_avail": [f"[{l:.4f}, {h:.4f}]" for l, h in zip(a.ci_low, a.ci_high)],
    "ci_imputed": [f"[{l:.4f}, {h:.4f}]" for l, h in zip(m.ci_low, m.ci_high)],
    "ci_excl_zero_avail": a.ci_excludes_zero, "ci_excl_zero_imputed": m.ci_excludes_zero,
    "rank_available": a.abs_rank, "rank_imputed": m.abs_rank,
}).reset_index()
comp3["rank_changed"] = comp3.rank_available != comp3.rank_imputed
comp3.to_csv(os.path.join(OUT_DIR, "robustness_comparison.csv"), index=False)
display(comp3)


repro = res.equals(run_pipeline(SEED_DIGIT))
big = comp3.loc[comp3.change_in_diff.abs().idxmax()]
sign_feats = list(comp3.loc[comp3.sign_changed, "feature"])
rank_feats = list(comp3.loc[comp3.rank_changed, "feature"])

report = f"""SENSITIVITY REPORT
Student: {LAST_NAME} | ID: {STUDENT_ID}
SEED_DIGIT = {SEED_DIGIT} | ID_SUM = {ID_SUM}
Features: {cols3}
Bootstrap repetitions: {B} (200 + 10 x {SEED_DIGIT}), stratified by outcome, seed = {SEED_DIGIT}
Strategies: available-case analysis vs. median imputation (full-dataset medians)

Largest change in mean difference: {big.feature} ({big.change_in_diff:+.6f})
Features with changed difference sign: {sign_feats if sign_feats else 'None'}
Features with changed ranking: {rank_feats if rank_feats else 'None'}
Features whose 95% CI excludes zero (available / imputed):
  available: {list(comp3.loc[comp3.ci_excl_zero_avail,'feature'])}
  imputed  : {list(comp3.loc[comp3.ci_excl_zero_imputed,'feature'])}
Reproducibility (same seed, identical results): {repro}

Interpretation:
Median imputation pulls filled values toward each group's center, so mean differences and interval
widths can shift versus available-case analysis. Where the sign, CI-vs-zero conclusion and ranking
agree across strategies, the Fail-vs-Pass difference is robust to the preparation method; features
whose conclusions change should be treated as sensitive to missing-data handling.
"""
open(os.path.join(OUT_DIR, "sensitivity_report.txt"), "w").write(report)
print(report)


print("Selected Features                  :", cols3)
print("Bootstrap Repetitions              :", B)
print("Largest Change in Mean Difference  :", big.feature, round(big.change_in_diff, 6))
print("Features with Changed Sign         :", sign_feats or "None")
print("Features with Changed Ranking      :", rank_feats or "None")
print("Reproducibility Verified           :", repro)

Selected: ['F011', 'F254', 'F383', 'F469', 'F472'] | Bootstrap repetitions: 250


,feature,strategy,n_fail,n_pass,mean_diff,boot_mean_diff,ci_low,ci_high,ci_excludes_zero,abs_rank
0,F011,available_case,104,1461,-0.001630,-0.001540,-0.003599,0.000467,False,4
1,F254,available_case,104,1463,-0.000789,-0.000746,-0.002419,0.000896,False,5
2,F383,available_case,30,519,-0.142853,-0.142475,-0.292029,0.007181,False,3
3,F469,available_case,104,1457,0.665356,0.665841,-0.054488,1.641126,False,1
4,F472,available_case,104,1456,0.233118,0.198215,-8.784782,9.459493,False,2
5,F011,median_imputation,104,1463,-0.001632,-0.001600,-0.003459,0.000322,False,4
6,F254,median_imputation,104,1463,-0.000789,-0.000768,-0.002533,0.000908,False,5
7,F383,median_imputation,104,1463,-0.052891,-0.052260,-0.117398,-0.011120,True,3
8,F469,median_imputation,104,1463,0.665953,0.636079,-0.113327,1.617817,False,1
9,F472,median_imputation,104,1463,0.231289,0.224402,-9.924434,8.279872,False,2


,feature,diff_available,diff_imputed,change_in_diff,sign_changed,ci_avail,ci_imputed,ci_excl_zero_avail,ci_excl_zero_imputed,rank_available,rank_imputed,rank_changed
0,F011,-0.001630,-0.001632,-0.000002,False,"[-0.0036, 0.0005]","[-0.0035, 0.0003]",False,False,4,4,False
1,F254,-0.000789,-0.000789,0.000000,False,"[-0.0024, 0.0009]","[-0.0025, 0.0009]",False,False,5,5,False
2,F383,-0.142853,-0.052891,0.089962,False,"[-0.2920, 0.0072]","[-0.1174, -0.0111]",False,True,3,3,False
3,F469,0.665356,0.665953,0.000597,False,"[-0.0545, 1.6411]","[-0.1133, 1.6178]",False,False,1,1,False
4,F472,0.233118,0.231289,-0.001829,False,"[-8.7848, 9.4595]","[-9.9244, 8.2799]",False,False,2,2,False


SENSITIVITY REPORT
Student: Lancion | ID: TUPM-26-1845
SEED_DIGIT = 5 | ID_SUM = 26
Features: ['F011', 'F254', 'F383', 'F469', 'F472']
Bootstrap repetitions: 250 (200 + 10 x 5), stratified by outcome, seed = 5
Strategies: available-case analysis vs. median imputation (full-dataset medians)

Largest change in mean difference: F383 (+0.089962)
Features with changed difference sign: None
Features with changed ranking: None
Features whose 95% CI excludes zero (available / imputed):
  available: []
  imputed  : ['F383']
Reproducibility (same seed, identical results): True

Interpretation:
Median imputation pulls filled values toward each group's center, so mean differences and interval
widths can shift versus available-case analysis. Where the sign, CI-vs-zero conclusion and ranking
agree across strategies, the Fail-vs-Pass difference is robust to the preparation method; features
whose conclusions change should be treated as sensitive to missing-data handling.

Selected Features            